In [7]:
from helpers import *
import pandas as pd

folder = "10991360/"

csv_df = pd.read_csv("data_preparation_output.csv")

display(csv_df)

,customer_name,customer_branch,file_name,order_quantity,id
0,MetalForm SA,Frankfurt Branch,Geo (1).DXF,1600,1
1,MetalForm SA,Frankfurt Branch,Geo (57).DXF,500,1
2,MetalForm SA,Frankfurt Branch,Geo (37).DXF,1800,1
3,MetalForm SA,Frankfurt Branch,Geo (44).DXF,700,1
4,MetalForm SA,Frankfurt Branch,Geo (19).DXF,1450,1
...,...,...,...,...,...
79,NaN,NaN,Geo (72).DXF,0,6
80,NaN,NaN,Geo (16).DXF,0,6
81,NaN,NaN,Geo (18).DXF,0,6
82,NaN,NaN,Geo (45).DXF,0,6


In [8]:
cutting_speed_mm_min = 1000 # mm/min
offset_general_s = 120 #s 
offset_time_smartgate_s = 2 #s
offset_time_sortmaster_s = 5 #s
offset_time_manually_s = 20 #s
offset_time_hole_s = 3 #s

smartgate_min = (30, 30) # min
smartgate_max = (160, 160) # max

sortmaster_min = (60, 90) # min
sortmaster_max = (1500, 2000) # max

def get_cutting_time(path):
    cutting_time_s = get_cutting_length(path) / (cutting_speed_mm_min / 60)

    extraction_method = get_extraction_method(path)

    if extraction_method == "Gate + Master":
        offset_extraction_method_s = min(offset_time_smartgate_s, offset_time_sortmaster_s)
    elif extraction_method == "Master":
        offset_extraction_method_s = offset_time_sortmaster_s
    elif extraction_method == "Gate":
        offset_extraction_method_s = offset_time_smartgate_s
    else:
        offset_extraction_method_s = offset_time_manually_s

    offset_time_total_s = offset_general_s + offset_extraction_method_s + get_number_of_holes(path) * offset_time_hole_s

    return cutting_time_s + offset_time_total_s

def get_fit(object, min_opening, max_opening):
    ob_1, ob_2 = sorted(object)
    min_1, min_2 = sorted(min_opening)
    max_1, max_2 = sorted(max_opening)

    return min_1 <= ob_1 <= max_1 and min_2 <= ob_2 <= max_2

def get_extraction_method(path):

    bounding_box = tuple(sorted(get_dxf_bound(path)))

    if get_fit(bounding_box, sortmaster_min, sortmaster_max) and get_fit(bounding_box, smartgate_min, smartgate_max):
        return "Gate + Master"
    
    elif get_fit(bounding_box, smartgate_min, smartgate_max):
        return "Gate"
    
    elif get_fit(bounding_box, sortmaster_min, sortmaster_max):
        return "Master"
    
    else:
        return "Manual"

def get_bounding_box_area(path):
    bounding_box = get_dxf_bound(path)
    return bounding_box[0] * bounding_box[1]

def get_automation_index(extraction_method):

    if extraction_method != "Manual":
        return 1
    else:
        return 0

csv_df['path'] = folder + csv_df['file_name']

csv_df[['bounding_box_width_mm', 'bounding_box_height_mm']] = csv_df['path'].apply(lambda p: pd.Series(get_dxf_bound(p)))

csv_df['bounding_box_area_mm^2'] = csv_df['bounding_box_width_mm'] * csv_df['bounding_box_height_mm']
csv_df['part_area_mm^2'] = csv_df['path'].apply(get_part_area)
csv_df['aspect_ratio'] = csv_df['bounding_box_height_mm'] / csv_df['bounding_box_width_mm']
csv_df['rectangularity_score'] = csv_df['part_area_mm^2'] / csv_df['bounding_box_area_mm^2']

csv_df['extraction_method'] = csv_df["path"].apply(get_extraction_method)
csv_df['automation_index'] = csv_df["extraction_method"].map(get_automation_index)

csv_df['number_of_holes'] = csv_df["path"].apply(get_number_of_holes)
csv_df['cutting_length_mm'] = csv_df["path"].apply(get_cutting_length)
csv_df['cutting_time_s'] = csv_df["path"].apply(get_cutting_time)

In [9]:
display(csv_df)

,customer_name,customer_branch,file_name,order_quantity,id,path,bounding_box_width_mm,bounding_box_height_mm,bounding_box_area_mm^2,part_area_mm^2,aspect_ratio,rectangularity_score,extraction_method,automation_index,number_of_holes,cutting_length_mm,cutting_time_s
0,MetalForm SA,Frankfurt Branch,Geo (1).DXF,1600,1,10991360/Geo (1).DXF,1600.219048,305.139765,4.882905e+05,4.013592e+04,0.190686,0.082197,Master,1,0,3438.024977,331.281499
1,MetalForm SA,Frankfurt Branch,Geo (57).DXF,500,1,10991360/Geo (57).DXF,1453.993892,1104.818310,1.606399e+06,1.520242e+06,0.759851,0.946366,Master,1,27,8584.200459,721.052028
2,MetalForm SA,Frankfurt Branch,Geo (37).DXF,1800,1,10991360/Geo (37).DXF,1947.204525,1134.770001,2.209629e+06,1.517175e+06,0.582769,0.686620,Master,1,31,8330.442132,717.826528
3,MetalForm SA,Frankfurt Branch,Geo (44).DXF,700,1,10991360/Geo (44).DXF,1097.000000,172.968300,1.897462e+05,1.894595e+05,0.157674,0.998489,Master,1,4,2609.913223,293.594793
4,MetalForm SA,Frankfurt Branch,Geo (19).DXF,1450,1,10991360/Geo (19).DXF,1152.727829,314.994358,3.631028e+05,3.631028e+05,0.273260,1.000000,Master,1,0,2935.444373,301.126662
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
79,NaN,NaN,Geo (72).DXF,0,6,10991360/Geo (72).DXF,150.951745,150.951745,2.278643e+04,4.422425e+03,1.000000,0.194082,Gate + Master,1,1,885.706561,178.142394
80,NaN,NaN,Geo (16).DXF,0,6,10991360/Geo (16).DXF,400.349160,122.891989,4.919970e+04,2.972469e+04,0.306962,0.604164,Master,1,13,1460.629485,251.637769
81,NaN,NaN,Geo (18).DXF,0,6,10991360/Geo (18).DXF,100.159680,32.996556,3.304924e+03,3.297083e+03,0.329440,0.997627,Gate,1,0,261.162026,137.669722
82,NaN,NaN,Geo (45).DXF,0,6,10991360/Geo (45).DXF,1960.771357,708.272253,1.388760e+06,1.164195e+06,0.361221,0.838298,Master,1,31,8424.152782,723.449167
